In [32]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [33]:
import copy
import sys
import os

# Add the project root (i.e., the parent of 'experiments/' and 'src_torch/')
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [40]:
from src.data_utils import EmbeddingDatasetExtractor, get_batch
from src.regulariser import (
    L2Regulariser,
    L1Regulariser,
    UnbiasRegulariser,
    MultiRegulariser,
)
from src.trainer import EWCTrainer

import torch
import sklearn
from collections import defaultdict

In [35]:
DATASET_EXTRACTOR = EmbeddingDatasetExtractor("data/frozen_embeddings")
VALIDATION_RATIO = 0.1
MINIMUM_ITER_FINETUNING = 250000
initial_dataset_name = "jigsaw"

HYPERPARAMETERS = {
        "n_epochs": 1,
        "batchsize": 64,
        "learning_rate": 0.0005,
        "weight_decay": 0.000,
        "unbias_lambda": 0.0,
        "l1_lambda": 0.0,
        "l2_lambda": 0.01,
        "lmbd": 1e2
    }

In [38]:
def get_model(input_dim: int, seed=0, device="cuda", output_dim=10, head=None):
    torch.manual_seed(seed)
    model = torch.nn.Sequential(
        torch.nn.Linear(input_dim, output_dim),
    ).to(device)
    if head is not None:
        model.append(head)
    return model

def evaluate(model: torch.nn.Sequential, dataset_name: str, llm_name: str, seed: int = 42):
    _, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name, dataset_name, val_ratio=VALIDATION_RATIO, balance=False, use_cache=True
    )
    X, y = get_batch(dataset_val, batch_size=len(dataset_val), seed=seed)
    model.eval()
    with torch.no_grad():
        preds = torch.argmax(model(X), dim=1)
        accuracy = ((preds == y).sum()/y.shape[0]).item()
        balanced_accuracy = sklearn.metrics.balanced_accuracy_score(
            y.cpu().numpy(), preds.cpu().numpy()
        )
    return accuracy, balanced_accuracy

def head_train(
    dataset_name: str, finetuning_dataset: str, llm_name: str, balance: bool = True, lmbd: int = 1e6, seed: int = 0
):
    dataset_train, dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name,
        dataset_name,
        val_ratio=VALIDATION_RATIO,
        balance=balance,
        use_cache=True,
        seed=seed
    )
    fine_dataset_train, fine_dataset_val = DATASET_EXTRACTOR.get_embedding_dataset(
        llm_name,
        finetuning_dataset,
        val_ratio=VALIDATION_RATIO,
        balance=balance,
        use_cache=True,
        seed=seed
    )
    X, _ = get_batch(dataset_train, 100, seed=seed)
    model = get_model(X.shape[1], output_dim=2, seed=seed)
    hyperparameters = HYPERPARAMETERS

    l2 = L2Regulariser(hyperparameters["l2_lambda"])
    l1 = L1Regulariser(hyperparameters["l1_lambda"])
    unbias = UnbiasRegulariser(hyperparameters["unbias_lambda"])
    regulariser = MultiRegulariser([l2, l1, unbias])

    trainer = EWCTrainer(model, paradigm="CIL", seed=seed, lmbd=lmbd)

    trainer.train(
        dataset_train,
        dataset_val,
        epochs=hyperparameters["n_epochs"],
        batch_size=hyperparameters["batchsize"],
        lr=hyperparameters["learning_rate"],
        weight_decay=hyperparameters["weight_decay"],
        regulariser=regulariser,
        val_freq=500,
    )
    
    trainer.train(
        fine_dataset_train,
        fine_dataset_val,
        epochs=hyperparameters["n_epochs"],
        batch_size=hyperparameters["batchsize"],
        lr=hyperparameters["learning_rate"],
        weight_decay=hyperparameters["weight_decay"],
        regulariser=regulariser,
        val_freq=500,
    )

    return trainer, copy.deepcopy(trainer.model)

In [41]:
SEED = 0
lmbds = [1, 1e2, 1e4, 1e8, 1e12]
finetune_dataset_names = ["tweets_hate_speech_detection", "civil_comments", "hatemoji", "sbic", "hatecheck"]
m2_bert = "m2-bert-80M-2k-retrieval"
accs = defaultdict(list)
for lmbd in lmbds:
    print(lmbd)
    avg_accs = []
    for name in finetune_dataset_names:
        _, model = head_train(initial_dataset_name, name, m2_bert, seed=SEED, lmbd=lmbd)
        accs_new_dataset = evaluate(
                model, name, m2_bert, seed=SEED
        )
        accs_initial_dataset = evaluate(
            model, initial_dataset_name, m2_bert, seed=SEED
        )
        avg_accs.append((accs_initial_dataset[1] + accs_new_dataset[1]) / 2)
        print("-"*20)
        print(f"    |--> Accuracies on new dataset (SD/Macro) {accs_new_dataset}")
        print(f"    |--> Accuracies on initial dataset {initial_dataset_name} (SD/Macro) {accs_initial_dataset}")
        print(f"    Avg: {avg_accs}")
        print("-"*20)
    accs[lmbd] = avg_accs

1


Training Epochs: 100%|███████████████████████████████████████████████████████████████████████| 1/1 [00:05<00:00,  5.80s/it, val_loss=0.4450, val_acc=0.7902]


--------------------
    |--> Accuracies on new dataset (SD/Macro) (0.8044430613517761, 0.8093077809909446)
    |--> Accuracies on initial dataset jigsaw (SD/Macro) (0.7492009401321411, 0.7346885729359212)
    Avg: [0.7719981769634329]
--------------------


Training Epochs: 100%|███████████████████████████████████████████████████████████████████████| 1/1 [00:54<00:00, 54.22s/it, val_loss=0.5253, val_acc=0.7435]


--------------------
    |--> Accuracies on new dataset (SD/Macro) (0.7124888896942139, 0.7366798091663715)
    |--> Accuracies on initial dataset jigsaw (SD/Macro) (0.8329259753227234, 0.8281033390207979)
    Avg: [0.7719981769634329, 0.7823915740935847]
--------------------


...

--------------------
    |--> Accuracies on new dataset (SD/Macro) (0.6157760620117188, 0.5841586352054946)
    |--> Accuracies on initial dataset jigsaw (SD/Macro) (0.8599987030029297, 0.8396272548738508)
    Avg: [0.7234182021687727, 0.7750732810135719, 0.7118929450396727]
--------------------


Training Epochs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:03<00:00,  3.44s/it]


--------------------
    |--> Accuracies on new dataset (SD/Macro) (0.6276485323905945, 0.6333706355193414)
    |--> Accuracies on initial dataset jigsaw (SD/Macro) (0.8599987030029297, 0.8396272548738508)
    Avg: [0.7234182021687727, 0.7750732810135719, 0.7118929450396727, 0.7364989451965961]
--------------------


Training Epochs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.15s/it]


--------------------
    |--> Accuracies on new dataset (SD/Macro) (0.6241611242294312, 0.5349114355963671)
    |--> Accuracies on initial dataset jigsaw (SD/Macro) (0.860186755657196, 0.8397313420294957)
    Avg: [0.7234182021687727, 0.7750732810135719, 0.7118929450396727, 0.7364989451965961, 0.6873213888129315]
--------------------


In [45]:
for keys, vals in accs.items():
    print(keys, sum(vals) / len(vals))

1 0.7476771979876093
100.0 0.7496533030491676
10000.0 0.7535406431123317
100000000.0 0.7267538378847189
1000000000000.0 0.726840952446309
